# 1D Axial Dispersion Model — Reproduction Notebook

**Thesis:** *Axial Dispersion and Viscosity-Dependent Effects in Models of Glucose Absorption in the Human Small Intestine*  
Alexander Møller Rivera, École Polytechnique (2025/2026)

This notebook reproduces all five figures from the thesis in order:
- **Figure 2** — Mass conservation error
- **Figure 3** — Ideal PFR absorption curves
- **Figure 4** — ADM absorption at physiological Pe (water-like)
- **Figure 5** — Absorbed fraction vs viscosity (three configurations)
- **Figure 6** — Isolated dispersion contribution ΔF

---
### Governing equations (dimensionless, Section 4.5)

$$\frac{dS'_s}{d\tau} = -\tau_\text{emptying}\, S'_s$$

$$\frac{\partial S'}{\partial \tau} = -\frac{\partial S'}{\partial \xi} + \frac{1}{Pe}\frac{\partial^2 S'}{\partial \xi^2} - \tau_R \frac{S'}{K_{mII}+S'} + \tau_\text{emptying}\,S'_s\,\delta(\xi)$$

$$\frac{\partial G'}{\partial \tau} = -\frac{\partial G'}{\partial \xi} + \frac{1}{Pe}\frac{\partial^2 G'}{\partial \xi^2} + \tau_R \frac{S'}{K_{mII}+S'} - \tau_\text{transfer}\,G'$$

In [ ]:
import sys, os
# Make src importable when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from src.solver    import simulate
from src.transport import (
    peclet, transport_params, tau_emptying_from_halflife,
    tau_R_from_Vmax, hours_to_tau, tau_to_hours,
)
from src.model      import build_rhs, mass_balance
from src.parameters import (
    NX, RTOL, ATOL, RTOL_SWEEP, ATOL_SWEEP,
    M0_g, MW_AGU, Km_mM, rm, L, u,
)
from src.plotting import (
    plot_mass_conservation, plot_pfr_absorption, plot_adm_absorption,
    plot_absorption_vs_viscosity, plot_dispersion_contribution,
)

import math
A    = math.pi * rm**2
C0_mM = (M0_g / MW_AGU) / (A * L * 1e3)

VMAX_LIST  = [4.0, 9.0, 16.0]   # mM/min
T_HALF_MIN = 20.0                # gastric half-emptying time [min]
MU_WATER   = 1e-3                # Pa·s

FIGURES_DIR = os.path.join('..', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

%matplotlib inline
print('Setup complete.')

---
## Figure 2 — Mass conservation error

The conservative flux-divergence scheme enforces discrete mass balance by construction.  
Any violation of $M(\tau)=1$ is therefore attributable entirely to the time-integration error.  
We expect $|M(\tau)-1| < 10^{-8}$ for all viscosities.

In [ ]:
from scipy.integrate import solve_ivp

mu_list_fig2 = [1e-3, 1e-2, 1e-1, 1e0, 1e1]
results_fig2 = {}

for mu in mu_list_fig2:
    tau_tr = transport_params(mu)[0]
    tau_em = tau_emptying_from_halflife(T_HALF_MIN)
    tau_R  = tau_R_from_Vmax(9.0, C0_mM)
    KmII   = Km_mM / C0_mM
    Pe_val = peclet(mu)
    tau_end = hours_to_tau(3.0)
    dxi = 1.0 / (NX - 1)

    rhs, y0, n = build_rhs(tau_tr, tau_em, tau_R, KmII, Pe_val, NX)
    t_eval = tau_end * np.linspace(0, 1, 800)
    sol = solve_ivp(rhs, (0, tau_end), y0, t_eval=t_eval,
                    method='BDF', rtol=RTOL, atol=ATOL)

    M_series = mass_balance(sol.y, n, dxi)
    label = rf'$\mu={mu:.0e}$ Pa·s  ($Pe={Pe_val:.0f}$)'
    results_fig2[label] = dict(
        t_h=tau_to_hours(sol.t),
        mb_error_series=np.abs(M_series - 1.0)
    )
    print(f'  μ={mu:.0e}  Pe={Pe_val:.1f}  max|M-1|={np.max(np.abs(M_series-1)):.2e}')

fig2 = plot_mass_conservation(
    results_fig2,
    save_path=os.path.join(FIGURES_DIR, 'fig2_mass_conservation.png')
)
plt.show()

---
## Figure 3 — Ideal PFR absorption curves

The plug-flow limit ($Pe \to \infty$) is approximated by setting `Pe_override=1e9`.  
These reproduce the Moxon et al. (2016) reference curves (Figure 3 of thesis).

In [ ]:
results_pfr = {}
for vmax in VMAX_LIST:
    res = simulate(MU_WATER, vmax, T_HALF_MIN, Pe_override=1e9,
                   nx=NX, rtol=RTOL, atol=ATOL)
    results_pfr[vmax] = res
    print(f'Vmax={vmax}  absorbed(3h)={res["absorbed_g"][-1]:.1f} g  mb_err={res["mb_error"]:.2e}')

fig3 = plot_pfr_absorption(
    results_pfr,
    save_path=os.path.join(FIGURES_DIR, 'fig3_pfr_absorption.png')
)
plt.show()

---
## Figure 4 — ADM at physiological Pe (water-like chyme, $\mu = 10^{-3}$ Pa·s)

The ADM at the lowest viscosity (water, $\alpha=1$, full Taylor–Aris) should be nearly
indistinguishable from the PFR curves above, confirming that the dispersion term vanishes
in the correct limit and that the implementation is consistent.

In [ ]:
Pe_water = peclet(MU_WATER, alpha=1.0)
print(f'Pe at μ=10⁻³ Pa·s (full cylinder, α=1): {Pe_water:.2f}')

results_adm = {}
for vmax in VMAX_LIST:
    res = simulate(MU_WATER, vmax, T_HALF_MIN, alpha=1.0,
                   nx=NX, rtol=RTOL, atol=ATOL)
    results_adm[vmax] = res
    print(f'Vmax={vmax}  absorbed(3h)={res["absorbed_g"][-1]:.1f} g  mb_err={res["mb_error"]:.2e}')

fig4 = plot_adm_absorption(
    results_adm, Pe=Pe_water,
    save_path=os.path.join(FIGURES_DIR, 'fig4_adm_absorption.png')
)
plt.show()

---
## Figure 5 — Absorbed fraction at t = 3 h vs viscosity

Three model configurations (Section 6.2):
- **α=1**: full Taylor–Aris dispersion
- **α=600**: dispersion suppressed → plug-flow reference  
- **Pe=5000, α=1**: high-Pe override isolating wall-absorption pathway

The near-complete overlap of all three curves demonstrates that **wall mass transfer, not axial dispersion**, is the dominant determinant of total absorption across the full viscosity range.

In [ ]:
MU_SWEEP        = np.logspace(-3, 1, 20)
VMAX_SWEEP      = 9.0
TAU_EMPTY_SWEEP = 10.0   # dimensionless

# Back-calculate gastric half-time from tau_emptying = g*L/u, g = ln2/(t_half*60)
t_half_sweep = (np.log(2) / (TAU_EMPTY_SWEEP * u / L)) / 60.0  # [min]

def absorbed_at_3h(mu, alpha=1.0, Pe_override=None):
    res = simulate(mu, VMAX_SWEEP, t_half_sweep,
                   alpha=alpha, Pe_override=Pe_override,
                   nx=NX, rtol=RTOL_SWEEP, atol=ATOL_SWEEP)
    return float(np.interp(3.0, res['t_h'], res['absorbed_fraction']))

F_full   = np.array([absorbed_at_3h(mu, alpha=1.0)            for mu in MU_SWEEP])
F_pfr    = np.array([absorbed_at_3h(mu, alpha=600.0)          for mu in MU_SWEEP])
F_highpe = np.array([absorbed_at_3h(mu, alpha=1.0, Pe_override=5000.) for mu in MU_SWEEP])

print('μ [Pa·s]   F_full   F_pfr   F_highpe')
for mu, fa, fp, fh in zip(MU_SWEEP, F_full, F_pfr, F_highpe):
    print(f'  {mu:.2e}    {fa:.3f}   {fp:.3f}   {fh:.3f}')

fig5 = plot_absorption_vs_viscosity(
    MU_SWEEP, F_full, F_pfr, F_highpe,
    save_path=os.path.join(FIGURES_DIR, 'fig5_absorption_vs_viscosity.png')
)
plt.show()

---
## Figure 6 — Isolated dispersion contribution ΔF

$$\Delta F = F_{\alpha=1} - F_{\text{PFR}}$$

At low viscosity ($\mu = 10^{-3}$ Pa·s), axial dispersion increases absorption by
approximately 6 percentage points (~3 g for a 50 g starch bolus).  
Above $\mu \approx 10^{-1}$ Pa·s the contribution is negligible.

In [ ]:
delta_F = F_full - F_pfr

print(f'Max ΔF = {delta_F.max():.4f}  at μ = {MU_SWEEP[delta_F.argmax()]:.2e} Pa·s')
print(f'Equivalent glucose: {delta_F.max() * M0_g:.1f} g from {M0_g:.0f} g starch bolus')

fig6 = plot_dispersion_contribution(
    MU_SWEEP, delta_F,
    save_path=os.path.join(FIGURES_DIR, 'fig6_dispersion_contribution.png')
)
plt.show()

---
## Summary

| Figure | Key result |
|--------|------------|
| 2 | Mass conservation error < 10⁻⁸ for all viscosities |
| 3 | PFR reference: ~39 g absorbed at 3h for V_max=16 mM/min |
| 4 | ADM at water viscosity matches PFR — plug-flow limit recovered |
| 5 | All three configurations nearly identical — wall transfer dominates |
| 6 | Max ΔF ≈ 0.065 (6.5%) at water viscosity; negligible above 0.1 Pa·s |

**Conclusion:** The plug-flow approximation of Moxon et al. (2016) is quantitatively accurate across the entire viscosity range relevant to dietary fibre interventions.